In [1]:
#!/usr/bin/env python3
"""
Merge sysadmin comment CSV shards into one file, with:
  - duplicate comment_id detection/removal (keep first occurrence)
  - date coverage report per input file (min/max date)
  - gap check across the merged timeline
  - final row count + written output

Usage:
    python3 merge_sysadmin_comments.py

Edit INPUT_FILES below to match your actual local paths/filenames.
Adjust ID_COL and DATE_COL if your CSV headers differ.
"""

import pandas as pd
import glob
import os
import sys

# ---- CONFIG: edit these to match your local setup ----------------------

# Option A: explicit list (safest, avoids accidentally pulling in old/retired files)
INPUT_FILES = [
    "sysadmin_comments_filteredQ1.csv",   # 2018-01-01 to 2018-12-31
    "sysadmin_comments_filteredQ2.csv",   # 2019-01-01 to 2019-12-31
    "sysadmin_comments_filteredQ3.csv",   # 2020-01-01 to 2021-06-30
    "sysadmin_comments_filtered2021Q10.csv",  # 2021-07-01 to 2021-12-31
    "sysadmin_comments_filtered2022Q11.csv",  # 2022-01-01 to 2022-05-30
    "sysadmin_comments_filtered2022Q12.csv",  # 2022-05-31 to 2022-12-31
    "sysadmin_comments_filtered2023Q13.csv",  # 2023-01-01 to 2023-05-31
    "sysadmin_comments_filtered2023Q14.csv",  # 2023-06-01 to 2023-12-31
    "sysadmin_comments_filtered2024Q15.csv",  # 2024-01-01 to 2024-05-31
    "sysadmin_comments_filtered2024Q16.csv",  # 2024-06-01 to 2024-12-31
    "sysadmin_comments_filtered2025Q17.csv",  # 2025-01-01 to 2025-12-31
    "sysadmin_comments_filtered2026Q18.csv",  # 2026-01-01 to 2026-04-30
]

OUTPUT_FILE = "filtered_raw_sysadmin_comments_new.csv"

# Column names -- adjust if your CSVs use different headers
ID_COL = "comment_id"      # unique identifier for a comment (e.g. Reddit comment id/fullname)
DATE_COL = "created_utc"   # timestamp column; can be unix epoch (int) or ISO string

# --------------------------------------------------------------------------


def load_and_tag(path):
    if not os.path.exists(path):
        print(f"  [MISSING] {path} not found -- skipping")
        return None
    df = pd.read_csv(path)
    df["__source_file"] = os.path.basename(path)
    return df


def to_datetime_col(df, date_col):
    """Handle both unix epoch and ISO string timestamps."""
    if pd.api.types.is_numeric_dtype(df[date_col]):
        return pd.to_datetime(df[date_col], unit="s", errors="coerce")
    return pd.to_datetime(df[date_col], errors="coerce")


def main():
    print("=" * 70)
    print("STEP 1: Loading input files")
    print("=" * 70)

    frames = []
    for path in INPUT_FILES:
        df = load_and_tag(path)
        if df is None:
            continue
        if ID_COL not in df.columns:
            print(f"  [WARN] {path}: missing column '{ID_COL}' -- check ID_COL config")
        if DATE_COL not in df.columns:
            print(f"  [WARN] {path}: missing column '{DATE_COL}' -- check DATE_COL config")
        frames.append(df)
        print(f"  Loaded {path}: {len(df):,} rows")

    if not frames:
        print("No files loaded. Check INPUT_FILES paths.")
        sys.exit(1)

    print()
    print("=" * 70)
    print("STEP 2: Per-file date coverage")
    print("=" * 70)

    coverage = []
    for df in frames:
        src = df["__source_file"].iloc[0]
        if DATE_COL in df.columns:
            dt = to_datetime_col(df, DATE_COL)
            dmin, dmax = dt.min(), dt.max()
            n_bad = dt.isna().sum()
            coverage.append((src, dmin, dmax, len(df)))
            bad_note = f" ({n_bad} unparseable dates)" if n_bad else ""
            print(f"  {src:45s} {dmin} -> {dmax}  [{len(df):,} rows]{bad_note}")
        else:
            print(f"  {src:45s} [no date column found]")

    print()
    print("=" * 70)
    print("STEP 3: Gap check across merged timeline")
    print("=" * 70)

    coverage_sorted = sorted(coverage, key=lambda x: x[1])
    gap_found = False
    for i in range(1, len(coverage_sorted)):
        prev_src, prev_min, prev_max, _ = coverage_sorted[i - 1]
        cur_src, cur_min, cur_max, _ = coverage_sorted[i]
        gap_days = (cur_min - prev_max).days
        if gap_days > 1:
            gap_found = True
            print(f"  [GAP] {gap_days} days between {prev_src} (ends {prev_max.date()}) "
                  f"and {cur_src} (starts {cur_min.date()})")
        elif gap_days < -1:
            print(f"  [OVERLAP] {prev_src} and {cur_src} overlap by {-gap_days} days "
                  f"({prev_max.date()} vs {cur_min.date()})")
    if not gap_found:
        print("  No gaps >1 day detected between consecutive files.")
    if coverage_sorted:
        overall_min = min(c[1] for c in coverage_sorted)
        overall_max = max(c[2] for c in coverage_sorted)
        print(f"\n  Overall merged range: {overall_min.date()} -> {overall_max.date()}")

    print()
    print("=" * 70)
    print("STEP 4: Concatenate + duplicate check")
    print("=" * 70)

    merged = pd.concat(frames, ignore_index=True)
    total_before = len(merged)
    print(f"  Total rows before dedup: {total_before:,}")

    if ID_COL in merged.columns:
        dupe_mask = merged.duplicated(subset=[ID_COL], keep="first")
        n_dupes = dupe_mask.sum()
        if n_dupes:
            dupe_preview = merged.loc[dupe_mask, [ID_COL, "__source_file"]].head(20)
            print(f"  Found {n_dupes:,} duplicate {ID_COL} values -- dropping (keep first occurrence)")
            print("  Sample duplicates (first 20):")
            print(dupe_preview.to_string(index=False))
        else:
            print(f"  No duplicate {ID_COL} values found.")
        merged = merged.drop_duplicates(subset=[ID_COL], keep="first")
    else:
        print(f"  [WARN] Cannot dedup -- '{ID_COL}' column not present. "
              f"Check ID_COL config against actual CSV headers: {list(merged.columns)}")

    total_after = len(merged)
    print(f"  Total rows after dedup: {total_after:,}  (removed {total_before - total_after:,})")

    merged = merged.drop(columns=["__source_file"])

    print()
    print("=" * 70)
    print("STEP 5: Write output")
    print("=" * 70)
    merged.to_csv(OUTPUT_FILE, index=False)
    print(f"  Wrote {len(merged):,} rows to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

STEP 1: Loading input files
  Loaded sysadmin_comments_filteredQ1.csv: 24,452 rows
  Loaded sysadmin_comments_filteredQ2.csv: 21,487 rows
  Loaded sysadmin_comments_filteredQ3.csv: 37,542 rows
  Loaded sysadmin_comments_filtered2021Q10.csv: 16,937 rows
  Loaded sysadmin_comments_filtered2022Q11.csv: 17,034 rows
  Loaded sysadmin_comments_filtered2022Q12.csv: 29,498 rows
  Loaded sysadmin_comments_filtered2023Q13.csv: 22,359 rows
  Loaded sysadmin_comments_filtered2023Q14.csv: 28,940 rows
  Loaded sysadmin_comments_filtered2024Q15.csv: 20,299 rows
  Loaded sysadmin_comments_filtered2024Q16.csv: 30,871 rows
  Loaded sysadmin_comments_filtered2025Q17.csv: 48,496 rows
  Loaded sysadmin_comments_filtered2026Q18.csv: 14,916 rows

STEP 2: Per-file date coverage
  sysadmin_comments_filteredQ1.csv              2018-01-02 17:53:08 -> 2018-12-31 23:41:46  [24,452 rows]
  sysadmin_comments_filteredQ2.csv              2019-01-01 00:04:54 -> 2019-12-31 23:45:13  [21,487 rows]
  sysadmin_comments_fil